In [ ]:
# Try different numbers of clusters
if len(embeddings) >= 5:
    n_clusters = min(max(3, len(embeddings) // 5), 10)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(embeddings)
    
    # Plot UMAP colored by cluster
    fig, ax = plt.subplots(figsize=(10, 7))
    for c in range(n_clusters):
        mask = cluster_labels == c
        ax.scatter(embedding_2d[mask, 0], embedding_2d[mask, 1], 
                   label=f"Cluster {c}", alpha=0.7, s=80)
    for i, title in enumerate(titles):
        ax.annotate(title[:15], (embedding_2d[i, 0], embedding_2d[i, 1]),
                     fontsize=7, alpha=0.6)
    ax.set_title(f"KMeans Clustering (k={n_clusters}) on Embedding Space")
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Print cluster contents
    for c in range(n_clusters):
        members = [tracks_with_emb[i] for i in range(len(tracks_with_emb)) if cluster_labels[i] == c]
        genres_in_cluster = [map_to_dj_genres(m.classification.genres).primary_genre for m in members]
        print(f"\nCluster {c} ({len(members)} tracks):")
        for m, g in zip(members, genres_in_cluster):
            print(f"  {m.title:<35} [{g}] BPM: {m.bpm}")
else:
    print("Need at least 5 tracks for clustering. Add more sample tracks.")

## 4. Clustering (KMeans)

Automatically cluster tracks and see if clusters align with genres.

In [ ]:
# Nearest neighbors for each track
for i, track in enumerate(tracks_with_emb):
    similarities = sim_matrix[i].copy()
    similarities[i] = -1  # exclude self
    top_5_idx = np.argsort(similarities)[::-1][:5]
    
    profile = map_to_dj_genres(track.classification.genres)
    print(f"\n{track.title} [{profile.primary_genre}] (BPM: {track.bpm}, Key: {track.key})")
    print(f"  Nearest neighbors:")
    for j in top_5_idx:
        neighbor = tracks_with_emb[j]
        n_profile = map_to_dj_genres(neighbor.classification.genres)
        print(f"    {similarities[j]:.3f}  {neighbor.title:<30} [{n_profile.primary_genre}] "
              f"BPM: {neighbor.bpm}, Key: {neighbor.key}")

## 3. Nearest Neighbor Test

For each track, show its 5 nearest neighbors by embedding similarity. Manually verify these make sense musically.

In [ ]:
# Cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

fig, ax = plt.subplots(figsize=(max(8, len(titles) * 0.5), max(8, len(titles) * 0.5)))
sns.heatmap(
    sim_matrix, xticklabels=[t[:20] for t in titles], yticklabels=[t[:20] for t in titles],
    cmap="YlOrRd", vmin=0, vmax=1, annot=len(titles) <= 20,
    fmt=".2f" if len(titles) <= 20 else "",
    ax=ax
)
ax.set_title("Track Similarity Matrix (Cosine Similarity of MusiCNN Embeddings)")
plt.tight_layout()
plt.show()

## 2. Cosine Similarity Heatmap

Pairwise similarity between all tracks. Similar tracks should form visible blocks.

In [ ]:
# Get DJ genre labels for coloring
dj_genres = []
dj_categories = []
for t in tracks_with_emb:
    profile = map_to_dj_genres(t.classification.genres)
    dj_genres.append(profile.primary_genre)
    dj_categories.append(get_genre_category(profile.primary_genre))

# UMAP reduction
reducer = umap.UMAP(n_neighbors=min(15, len(embeddings) - 1), min_dist=0.1, random_state=42)
embedding_2d = reducer.fit_transform(embeddings)

# Plot colored by DJ genre category
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# By category (broad)
unique_cats = sorted(set(dj_categories))
cat_colors = {cat: plt.cm.Set1(i / len(unique_cats)) for i, cat in enumerate(unique_cats)}
for cat in unique_cats:
    mask = [c == cat for c in dj_categories]
    axes[0].scatter(
        embedding_2d[mask, 0], embedding_2d[mask, 1],
        label=cat, alpha=0.7, s=60
    )
for i, title in enumerate(titles):
    axes[0].annotate(title[:15], (embedding_2d[i, 0], embedding_2d[i, 1]),
                      fontsize=7, alpha=0.6)
axes[0].set_title("UMAP — Colored by Genre Category")
axes[0].legend(fontsize=8)

# By specific DJ genre
unique_genres = sorted(set(dj_genres))
for genre in unique_genres:
    mask = [g == genre for g in dj_genres]
    axes[1].scatter(
        embedding_2d[mask, 0], embedding_2d[mask, 1],
        label=genre, alpha=0.7, s=60
    )
for i, title in enumerate(titles):
    axes[1].annotate(title[:15], (embedding_2d[i, 0], embedding_2d[i, 1]),
                      fontsize=7, alpha=0.6)
axes[1].set_title("UMAP — Colored by DJ Genre")
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## 1. UMAP Visualization

Reduce 200-dim embeddings to 2D for visualization. Color by primary DJ genre.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans, DBSCAN
import umap

from agent_dj.analyzer.track_store import TrackStore
from agent_dj.analyzer.genre_taxonomy import map_to_dj_genres, get_genre_category

sns.set_theme(style="whitegrid")

store = TrackStore("../tracks.db")
tracks = store.get_all()

# Filter tracks with embeddings
tracks_with_emb = [t for t in tracks if t.classification and t.classification.embedding]
print(f"Tracks with embeddings: {len(tracks_with_emb)} / {len(tracks)}")

# Build embedding matrix
embeddings = np.array([t.classification.embedding for t in tracks_with_emb])
titles = [t.title for t in tracks_with_emb]
print(f"Embedding shape: {embeddings.shape}")

# Experiment 03: Embedding Space Quality

The most important evaluation — do MusiCNN audio embeddings cluster musically similar tracks?

**Goals:**
- Visualize the embedding space (UMAP)
- Check if similar genres/moods naturally cluster
- Test nearest-neighbor quality: are the 5 nearest tracks actually similar?
- Determine if embeddings are good enough for track selection or need fine-tuning